In [ ]:
import sqlite3 as sql

conn = sql.connect("data/bookStore.db")
conn.execute("PRAGMA foreign_keys = ON")
cursor = conn.cursor()

# Categories table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Categories (
    ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Name TEXT UNIQUE NOT NULL
)
""")

# Books table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Books (
    ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Title TEXT NOT NULL,
    CategoryID INTEGER NOT NULL,
    Price_GBP REAL,
    Price_INR REAL,
    Star_rating INTEGER,
    in_stock INTEGER,
    FOREIGN KEY (CategoryID) REFERENCES Categories(ID)
)
""")

conn.commit()

print("Database and Tables(Books and Category) Created successfully")

In [ ]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
cursor.fetchall()

In [ ]:
import pandas as pd
books=pd.read_csv("data/clean_books.csv")
books.info()
Categories=books['Category'].unique()
Categories

In [ ]:
for Category in Categories:
    cursor.execute("INSERT OR IGNORE INTO Categories(Name) VALUES(?)",(Category,))
conn.commit()

In [ ]:
cursor.execute("PRAGMA table_info(Books)")
print(cursor.fetchall())

In [ ]:
cursor.execute('Select ID,Name from Categories')
category_map = {name: id for id, name in cursor.fetchall()}
books["CategoryID"] = books["Category"].map(category_map)

In [ ]:
books_sql = books[[
        "Title",
        "CategoryID",
        "Price_GBP",
        "Price_INR",
        "Star_rating",
        "In_stock"]]
books_sql.to_sql(
    "Books",
    conn,
    if_exists="append",
    index=False
)
conn.commit()


In [ ]:
cursor.execute("SELECT COUNT(*) FROM Books")
print(cursor.fetchone())

In [ ]:
queries = {}
outputs = {}

# Query 1
queries["Q1"] = """
SELECT * FROM Books;
"""
outputs["Q1"] = pd.read_sql(queries["Q1"], conn)

# Query 2
queries["Q2"] = """
SELECT * FROM Books
WHERE Star_rating >= 3;
"""
outputs["Q2"] = pd.read_sql(queries["Q2"], conn)

# Query 3
queries["Q3"] = """
SELECT DISTINCT Name
FROM Categories;
"""
outputs["Q3"] = pd.read_sql(queries["Q3"], conn)

# Query 4
queries["Q4"] = """
SELECT *
FROM Books
WHERE Price_GBP BETWEEN 20 AND 40;
"""
outputs["Q4"] = pd.read_sql(queries["Q4"], conn)

# Query 5
queries["Q5"] = """
SELECT *
FROM Books
WHERE Star_rating IN (4,5);
"""
outputs["Q5"] = pd.read_sql(queries["Q5"], conn)

# Query 6 (JOIN)
queries["Q6"] = """
SELECT
    b.Title,
    c.Name AS Category,
    b.Price_GBP,
    b.Price_INR,
    b.Star_rating,
    b.in_stock
FROM Books b
JOIN Categories c
ON b.CategoryID = c.ID
ORDER BY b.Title;
"""
outputs["Q6"] = pd.read_sql(queries["Q6"], conn)

# Query 7 : ORDER BY
queries["Q7"] = """
SELECT *
FROM Books
WHERE Star_rating >= 3
ORDER BY Star_rating DESC;
"""
outputs["Q7"] = pd.read_sql(queries["Q7"], conn)

# Query 8 : LIMIT + IN (subquery)
queries["Q8"] = """
SELECT *
FROM Books
WHERE ID IN (
    SELECT MIN(ID)
    FROM Books
    GROUP BY CategoryID, Star_rating
)
LIMIT 10;
"""
outputs["Q8"] = pd.read_sql(queries["Q8"], conn)

# Query 9 : IN (subquery)
queries["Q9"] = """
SELECT *
FROM Books
WHERE ID IN (
    SELECT MIN(ID)
    FROM Books
    GROUP BY CategoryID, Star_rating
);
"""
outputs["Q9"] = pd.read_sql(queries["Q9"], conn)

In [ ]:
for key in queries:
    print("=" * 70)
    print(key)
    print("SQL Query:")
    print(queries[key])

    print("\nOutput:")
    print(outputs[key])

In [ ]:
books_df = pd.read_sql("SELECT * FROM Books", conn)
categories_df = pd.read_sql("SELECT * FROM Categories", conn)

merge_df = pd.merge(
    books_df,
    categories_df,
    left_on="CategoryID",
    right_on="ID",
    how="inner"
)

merge_df = merge_df[[
    "Title",
    "Name",
    "Price_GBP",
    "Price_INR",
    "Star_rating",
    "in_stock"
]]

merge_df = merge_df.rename(columns={"Name": "Category"})
merge_df = merge_df.sort_values(
    ["Title", "Category"]
).reset_index(drop=True)

In [ ]:
sql_join = outputs["Q6"].copy()

In [ ]:
sql_join = sql_join.sort_values(
    ["Title", "Category"]
).reset_index(drop=True)
print("\nPandas Merge Result")
print(merge_df)
print("\nAre both outputs equivalent?")
print(sql_join.equals(merge_df))

In [ ]:
if not sql_join.equals(merge_df):
    print("\nDifferences:")
    print(sql_join.compare(merge_df))

In [ ]:
print(sql_join.columns)
print(merge_df.columns)